<a href="https://colab.research.google.com/github/anokhina-rgb/Google-Colabs/blob/main/berttmx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# === ВСЕ В ОДНОМУ КОДІ ДЛЯ GOOGLE COLAB ===

# 1. Встановлення необхідних бібліотек
!pip install -q python-docx sentence-transformers pandas openpyxl

# 2. Імпорт модулів та оголошення функцій
import os
import re
import math
import zipfile
import xml.etree.ElementTree as ET
from xml.dom import minidom
from google.colab import files

try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False

try:
    import docx
    HAS_DOCX = True
except ImportError:
    HAS_DOCX = False

try:
    from sentence_transformers import SentenceTransformer, util
    import torch
    HAS_BERT = True
except ImportError:
    HAS_BERT = False

def read_document(file_path):
    if file_path.lower().endswith('.docx'):
        if not HAS_DOCX:
            raise Exception("Бібліотеку python-docx не встановлено!")
        try:
            doc = docx.Document(file_path)
            lines = []
            for p in doc.paragraphs:
                txt = p.text.strip()
                if txt:
                    lines.append(txt)
            for table in doc.tables:
                for row in table.rows:
                    for cell in row.cells:
                        txt = cell.text.strip()
                        if txt and txt not in ["Текст оригіналу", "Текст перекладу"]:
                            lines.append(txt)
            return "\n".join(lines)
        except Exception as e:
            raise Exception(f"Помилка читання Word файлу: {e}")
    else:
        try:
            with open(file_path, "r", encoding="utf-8-sig") as f:
                return f.read()
        except Exception as e:
            raise Exception(f"Помилка читання текстового файлу: {e}")

def clean_and_split_sentences(text, remove_citations=True):
    if not text:
        return []
    text = re.sub(r'\r\n', '\n', text)
    raw_lines = [line.strip() for line in text.split('\n') if line.strip()]

    citation_pattern = re.compile(r'\s*\([А-ЯA-Za-zІіЇїЄєҐґа-яa-z\s.—–,\-]+\d{4}[^)]*?\)', re.UNICODE)
    general_parenthesis_pattern = re.compile(r'\s*\([^)]*?(?:ст\.\vert{}р\.\vert{}p\.\vert{}pp\.\vert{}20\d\d\vert{}19\d\d)[^)]*?\)', re.UNICODE)
    stray_number_pattern = re.compile(r'^\s*\d+[\.\)]+\s*$', re.UNICODE)
    prefix_number_pattern = re.compile(r'^\s*\d+[\.\)]+\s+', re.UNICODE)

    sentences = []
    for line in raw_lines:
        if line in ["Текст оригіналу", "Текст перекладу"]:
            continue
        if stray_number_pattern.match(line):
            continue
        cleaned_line = re.sub(r'\s+', ' ', line)
        cleaned_line = prefix_number_pattern.sub('', cleaned_line)

        if remove_citations:
            cleaned_line = citation_pattern.sub('', cleaned_line)
            cleaned_line = general_parenthesis_pattern.sub('', cleaned_line)
            cleaned_line = cleaned_line.strip()

        if cleaned_line:
            parts = re.split(r'(?<=[.!?])\s+(?=[A-Za-zА-Яа-їЄєІіЇїҐґ„"«\'"]?)', cleaned_line)
            for p in parts:
                p_clean = p.strip()
                if p_clean and not stray_number_pattern.match(p_clean):
                    sentences.append(p_clean)
    return sentences

def church_gale_align(source_sentences, target_sentences, mean_ratio=1.0, variance=6.8):
    I = len(source_sentences)
    J = len(target_sentences)
    source_lens = [len(s) for s in source_sentences]
    target_lens = [len(t) for t in target_sentences]

    dp = [[float('inf')] * (J + 1) for _ in range(I + 1)]
    backtrack = [[None] * (J + 1) for _ in range(I + 1)]
    dp[0][0] = 0.0

    bead_types = [(1, 1, 1.0), (1, 0, 450.0), (0, 1, 450.0), (2, 1, 110.0), (1, 2, 110.0), (2, 2, 220.0)]

    def calculate_cost(l1, l2):
        if l1 == 0 and l2 == 0:
            return 0.0
        c = mean_ratio
        l = (l1 + l2 / c) / 2.0
        if l == 0:
            return 0.0
        delta = l2 - l1 * c
        cost = (delta ** 2) / (2.0 * variance * l) if l > 0 else 0.0
        return max(0.0, cost)

    for i in range(I + 1):
        for j in range(J + 1):
            if dp[i][j] == float('inf'):
                continue
            for di, dj, base_penalty in bead_types:
                ni, nj = i + di, j + dj
                if ni <= I and nj <= J:
                    l1 = sum(source_lens[i:ni])
                    l2 = sum(target_lens[j:nj])
                    step_cost = calculate_cost(l1, l2) + base_penalty
                    total_cost = dp[i][j] + step_cost
                    if total_cost < dp[ni][nj]:
                        dp[ni][nj] = total_cost
                        backtrack[ni][nj] = (i, j)

    alignments = []
    i, j = I, J
    while i > 0 or j > 0:
        prev = backtrack[i][j]
        if prev is None:
            prev = (i - 1, j - 1) if i > 0 and j > 0 else ((i - 1, j) if i > 0 else (i, j - 1))
        pi, pj = prev
        src_segment = " ".join(source_sentences[pi:i]) if pi < i else ""
        tgt_segment = " ".join(target_sentences[pj:j]) if pj < j else ""
        alignments.append({"source": src_segment, "target": tgt_segment, "src_count": i - pi, "tgt_count": j - pj})
        i, j = pi, pj
    alignments.reverse()
    return alignments

def bert_semantic_align(source_sentences, target_sentences, threshold=0.35):
    if not HAS_BERT or not source_sentences or not target_sentences:
        return church_gale_align(source_sentences, target_sentences)
    try:
        model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
        src_emb = model.encode(source_sentences, convert_to_tensor=True)
        tgt_emb = model.encode(target_sentences, convert_to_tensor=True)
        sim_matrix = util.cos_sim(src_emb, tgt_emb)
        I, J = len(source_sentences), len(target_sentences)
        alignments = []
        i, j, window = 0, 0, 3

        while i < I and j < J:
            max_sim, best_j = -1.0, j
            for tj in range(j, min(J, j + window)):
                s = sim_matrix[i][tj].item()
                if s > max_sim:
                    max_sim, best_j = s, tj
            max_sim_i, best_i = -1.0, i
            for ti in range(i, min(I, i + window)):
                s = sim_matrix[ti][j].item()
                if s > max_sim_i:
                    max_sim_i, best_i = s, ti

            if max_sim >= threshold and best_j == j and max_sim >= max_sim_i:
                alignments.append({"source": source_sentences[i], "target": target_sentences[j], "src_count": 1, "tgt_count": 1})
                i, j = i + 1, j + 1
            elif max_sim_i > max_sim and max_sim_i >= threshold:
                src_seg = " ".join(source_sentences[i:best_i+1])
                alignments.append({"source": src_seg, "target": target_sentences[j], "src_count": best_i - i + 1, "tgt_count": 1})
                i, j = best_i + 1, j + 1
            elif max_sim >= threshold and best_j > j:
                tgt_seg = " ".join(target_sentences[j:best_j+1])
                alignments.append({"source": source_sentences[i], "target": tgt_seg, "src_count": 1, "tgt_count": best_j - j + 1})
                i, j = i + 1, best_j + 1
            else:
                if max_sim_i > max_sim:
                    alignments.append({"source": source_sentences[i], "target": "", "src_count": 1, "tgt_count": 0})
                    i += 1
                else:
                    alignments.append({"source": "", "target": target_sentences[j], "src_count": 0, "tgt_count": 1})
                    j += 1
        while i < I:
            alignments.append({"source": source_sentences[i], "target": "", "src_count": 1, "tgt_count": 0})
            i += 1
        while j < J:
            alignments.append({"source": "", "target": target_sentences[j], "src_count": 0, "tgt_count": 1})
            j += 1
        return alignments
    except Exception as e:
        print(f"BERT Error: {e}. Fallback to Church-Gale.")
        return church_gale_align(source_sentences, target_sentences)

def export_to_files(alignments, src_lang="en", tgt_lang="ukr"):
    os.makedirs("output", exist_ok=True)
    excel_path = "output/aligned_bitext.xlsx"
    tmx_path = "output/aligned_memory.tmx"

    if HAS_PANDAS:
        df = pd.DataFrame(alignments)
        df.columns = ["Оригінал (Source)", "Переклад (Target)", "К-ть реч. (Src)", "К-ть реч. (Tgt)"]
        try:
            df.to_excel(excel_path, index=False)
        except Exception:
            excel_path = "output/aligned_bitext.csv"
            df.to_csv(excel_path, index=False, encoding='utf-8-sig')

    tmx = ET.Element("tmx", version="1.4")
    ET.SubElement(tmx, "header", creationtool="BERTChurchGaleAligner", creationtoolversion="2.5", segtype="sentence", o_tmf="UTF-8", adminlang="en", srclang=src_lang)
    body = ET.SubElement(tmx, "body")
    for row in alignments:
        if not row['source'].strip() and not row['target'].strip():
            continue
        tu = ET.SubElement(body, "tu")
        tuv_src = ET.SubElement(tu, "tuv", attrib={"{http://www.w3.org/XML/1998/namespace}lang": src_lang})
        ET.SubElement(tuv_src, "seg").text = row['source']
        tuv_tgt = ET.SubElement(tu, "tuv", attrib={"{http://www.w3.org/XML/1998/namespace}lang": tgt_lang})
        ET.SubElement(tuv_tgt, "seg").text = row['target']

    rough_string = ET.tostring(tmx, encoding="utf-8")
    reparsed = minidom.parseString(rough_string)
    with open(tmx_path, "wb") as f:
        f.write(reparsed.toprettyxml(indent="  ", encoding="utf-8"))

    zip_path = "aligned_results.zip"
    with zipfile.ZipFile(zip_path, 'w') as zipf:
        zipf.write(excel_path, os.path.basename(excel_path))
        zipf.write(tmx_path, os.path.basename(tmx_path))
    return zip_path

# 3. Інтерактивний запуск і вивантаження
print("📁 Завантажте файл ОРИГІНАЛУ (.docx або .txt):")
uploaded_src = files.upload()
src_filename = list(uploaded_src.keys())[0]

print("\n📁 Завантажте файл ПЕРЕКЛАДУ (.docx або .txt):")
uploaded_tgt = files.upload()
tgt_filename = list(uploaded_tgt.keys())[0]

print(f"\n⚙️ Оброблено файли:\n• {src_filename}\n• {tgt_filename}")
source_text = read_document(src_filename)
target_text = read_document(tgt_filename)

src_sents = clean_and_split_sentences(source_text, remove_citations=True)
tgt_sents = clean_and_split_sentences(target_text, remove_citations=True)

if HAS_BERT:
    print("🚀 Використовується BERT семантичне вирівнювання...")
    alignments = bert_semantic_align(src_sents, tgt_sents)
else:
    print("⚙️ Використовується алгоритм Church-Gale...")
    alignments = church_gale_align(src_sents, tgt_sents, variance=6.8)

zip_file = export_to_files(alignments, src_lang="en", tgt_lang="ukr")
print(f"\n🎉 Вирівнювання завершено! Знайдено сегментів: {len(alignments)}")
print("📥 Автоматичне завантаження архіву `aligned_results.zip`...")
files.download(zip_file)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 18.5 MB/s eta 0:00:00
📁 Завантажте файл ОРИГІНАЛУ (.docx або .txt):


Saving Текст оригіналу_англ.docx to Текст оригіналу_англ.docx

📁 Завантажте файл ПЕРЕКЛАДУ (.docx або .txt):


Saving Текст перекладу_укр.docx to Текст перекладу_укр.docx

⚙️ Оброблено файли:
• Текст оригіналу_англ.docx
• Текст перекладу_укр.docx
🚀 Використовується BERT семантичне вирівнювання...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


🎉 Вирівнювання завершено! Знайдено сегментів: 70
📥 Автоматичне завантаження архіву `aligned_results.zip`...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>